In [33]:
# ============================================================
# D6 — Branch C: Normalised Markdown conversion
# 0. Imports and frozen experimental configuration
# ============================================================
!pip -q install pymupdf pymupdf4llm

import json
import hashlib
import platform
import re
import sys
import unicodedata

from collections import Counter
from datetime import datetime
from pathlib import Path

import fitz
import pymupdf4llm

from google.colab import files

DOCUMENT_ID = "D6"
DOCUMENT_NAME = "Microsoft FY24 Q1 Press Release"

BRANCH = "C"
BRANCH_NAME = "Deterministic normalisation"
PARENT_BRANCH = "B"

SOURCE_FORMAT = ".pdf"
EXPECTED_SOURCE_SHA256 = (
    "b3cf121cac62f6807d41e4e013a6edeebec0cbbf4f0ea2995e1fcc135abaac8d"
)

EXPECTED_PAGE_COUNT = 10
EXPECTED_RECORD_COUNT = 147

EXPECTED_CATEGORY_COUNTS = {
    "Narrative performance highlight": 24,
    "Financial performance reconciliation": 4,
    "Segment revenue reconciliation": 3,
    "Selected product and service reconciliation": 15,
    "Income statement": 19,
    "Comprehensive income statement": 6,
    "Balance sheet": 34,
    "Cash flow statement": 34,
    "Segment revenue and operating income": 8
}

EXPECTED_PART_COUNTS = {
    1: 46,
    2: 25,
    3: 34,
    4: 34,
    5: 8
}

EXPECTED_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Value 2023",
    "Value 2022",
    "GAAP YoY Change",
    "Constant Currency Impact",
    "Constant Currency YoY Change",
    "Unit",
    "Reporting Period",
    "Source Location"
]

STRING_OR_NULL_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Business Area",
    "Unit",
    "Reporting Period",
    "Source Location"
]

NUMERIC_OR_NULL_FIELDS = [
    "Value 2023",
    "Value 2022",
    "GAAP YoY Change",
    "Constant Currency Impact",
    "Constant Currency YoY Change"
]

# ------------------------------------------------------------
# Fields that must contain a value for every D6 record.
#
# Business Area is intentionally NOT included because several
# legitimate D6 observations are corporate/global observations
# for which no explicit business area is represented in source.
#
# A null Business Area is therefore schema-valid and may also be
# substantively correct.
# ------------------------------------------------------------

MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Statement or Section",
    "Metric",
    "Unit",
    "Reporting Period",
    "Source Location"
]

ALLOWED_CATEGORIES = set(EXPECTED_CATEGORY_COUNTS)

PART_PAGE_RANGES = {
    1: [1, 2, 3],
    2: [6, 7],
    3: [8],
    4: [9],
    5: [10]
}

EXPECTED_COMPONENT_MARKERS = {
    "quarterly_results":
        "Revenue was $56.5 billion",

    "business_highlights":
        "Business Highlights",

    "financial_performance_reconciliation":
        "Financial Performance Constant Currency Reconciliation",

    "segment_revenue_reconciliation":
        "Segment Revenue Constant Currency Reconciliation",

    "selected_product_reconciliation":
        "Selected Product and Service Revenue Constant Currency Reconciliation",

    "income_statements":
        "INCOME STATEMENTS",

    "comprehensive_income_statements":
        "COMPREHENSIVE INCOME STATEMENTS",

    "balance_sheets":
        "BALANCE SHEETS",

    "cash_flows_statements":
        "CASH FLOWS STATEMENTS",

    "segment_revenue_operating_income":
        "SEGMENT REVENUE AND OPERATING INCOME"
}

OUTPUT_DIR = Path("outputs_D6_branch_C")
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Document:",
    DOCUMENT_ID
)

print(
    "Branch:",
    BRANCH
)

print(
    "Parent branch:",
    PARENT_BRANCH
)

print(
    "Expected source pages:",
    EXPECTED_PAGE_COUNT
)


Document: D6
Branch: C
Parent branch: B
Expected source pages: 10


In [34]:
# ============================================================
# 1. Upload original D6 PDF + frozen Branch B artefacts
# ============================================================
uploaded = files.upload()
names = list(uploaded.keys())
pdfs=[Path(x) for x in names if x.lower().endswith(".pdf")]
mds=[Path(x) for x in names if x.lower().endswith(".md")]
jsons=[Path(x) for x in names if x.lower().endswith(".json")]
if len(pdfs)!=1 or len(mds)!=1 or len(jsons)!=1:
    raise ValueError("Upload exactly one PDF, one Branch B Markdown, and one Branch B conversion-integrity JSON.")
SOURCE_PATH=pdfs[0]
BRANCH_B_REPRESENTATION_PATH=mds[0]
BRANCH_B_CHECK_PATH=jsons[0]
print(SOURCE_PATH.name, BRANCH_B_REPRESENTATION_PATH.name, BRANCH_B_CHECK_PATH.name)


Saving D6_branch_B_structural_markdown.md to D6_branch_B_structural_markdown.md
Saving D6_branch_B_conversion_integrity.json to D6_branch_B_conversion_integrity.json
Saving D6 - Microsoft FY24 Q1 Press Release.pdf to D6 - Microsoft FY24 Q1 Press Release.pdf
D6 - Microsoft FY24 Q1 Press Release.pdf D6_branch_B_structural_markdown.md D6_branch_B_conversion_integrity.json


In [35]:
# ============================================================
# 2. Verify frozen source identity and Branch B integrity
# ============================================================
def sha256_file(path, chunk_size=1024*1024):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_text(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

if SOURCE_PATH.suffix.lower()!=SOURCE_FORMAT:
    raise ValueError("Unexpected source format.")
SOURCE_SHA256=sha256_file(SOURCE_PATH)
SOURCE_HASH_MATCH=(SOURCE_SHA256==EXPECTED_SOURCE_SHA256)
if not SOURCE_HASH_MATCH:
    raise ValueError("Uploaded PDF does not match the frozen D6 source identity.")

pdf_document=fitz.open(SOURCE_PATH)
if len(pdf_document)!=EXPECTED_PAGE_COUNT:
    raise ValueError(f"Expected {EXPECTED_PAGE_COUNT} pages; observed {len(pdf_document)}.")
if not all(page.get_text("text").strip() for page in pdf_document):
    raise ValueError("D6 should contain machine-readable text; OCR is not introduced.")

branch_b_check=json.loads(BRANCH_B_CHECK_PATH.read_text(encoding="utf-8"))
if branch_b_check.get("document_id")!=DOCUMENT_ID or branch_b_check.get("branch")!="B":
    raise ValueError("Wrong Branch B integrity artefact.")
if branch_b_check.get("source_sha256")!=SOURCE_SHA256:
    raise ValueError("Branch B was generated from a different source identity.")
if not branch_b_check.get("conversion_integrity_passed",False):
    raise ValueError("Branch B conversion integrity did not pass.")

SOURCE_B_MARKDOWN=BRANCH_B_REPRESENTATION_PATH.read_text(encoding="utf-8")
SOURCE_B_SHA256=sha256_text(SOURCE_B_MARKDOWN)
print("Source and Branch B parent verified.")


Source and Branch B parent verified.


In [36]:
# ============================================================
# 3. Reproduce exact Branch B complete Markdown
# ============================================================
page_chunks=pymupdf4llm.to_markdown(
    str(SOURCE_PATH), page_chunks=True, write_images=False, show_progress=True
)
if not isinstance(page_chunks,list) or len(page_chunks)!=EXPECTED_PAGE_COUNT:
    raise ValueError("Branch B reproduction returned unexpected page chunks.")

page_markdown={}
for page_number,chunk in enumerate(page_chunks,start=1):
    text=chunk.get("text","") if isinstance(chunk,dict) else str(chunk)
    if not text.strip():
        raise ValueError(f"Empty converted page {page_number}.")
    page_markdown[page_number]=text.rstrip()

sections=[
    "# D6 — Microsoft FY24 Q1 Press Release","",
    "> Complete structural conversion of the original 10-page PDF.",
    "> The same complete representation is used in all five extraction parts.",""
]
for page_number in range(1,EXPECTED_PAGE_COUNT+1):
    sections += [f"## Source Page {page_number}","",page_markdown[page_number],""]
REPRODUCED_BRANCH_B_MARKDOWN="\n".join(sections).rstrip()+"\n"
REPRODUCED_B_SHA256=sha256_text(REPRODUCED_BRANCH_B_MARKDOWN)
print("Reproduced B SHA-256:",REPRODUCED_B_SHA256)


Parsing 10 pages of 'D6 - Microsoft FY24 Q1 Press Release.pdf'...


100%|██████████| 10/10 [00:12<00:00,  1.22s/it]


=== Document parser messages ===
Using Tesseract for OCR processing.


Generating markdown text...


100%|██████████| 10/10 [00:00<00:00, 2494.98it/s]

Reproduced B SHA-256: fc2716f7590f8fdb9eca5b304dfe202670682ff697bfb80efa75f2bf67e51d46


In [37]:
# ============================================================
# 4. Exact Branch B parent equivalence
# ============================================================
PARENT_EQUIVALENCE_PASSED=(REPRODUCED_BRANCH_B_MARKDOWN==SOURCE_B_MARKDOWN)
parent_check={
    "document_id":DOCUMENT_ID,"branch":BRANCH,"parent_branch":PARENT_BRANCH,
    "source_sha256":SOURCE_SHA256,
    "branch_B_conversion_integrity_passed":bool(branch_b_check.get("conversion_integrity_passed",False)),
    "uploaded_branch_B_sha256":SOURCE_B_SHA256,
    "reproduced_branch_B_sha256":REPRODUCED_B_SHA256,
    "branch_B_representation_exactly_reproduced":PARENT_EQUIVALENCE_PASSED,
    "source_page_count":len(pdf_document),
    "parent_equivalence_passed":PARENT_EQUIVALENCE_PASSED
}
PARENT_CHECK_PATH=OUTPUT_DIR/"D6_branch_C_parent_B_equivalence_check.json"
PARENT_CHECK_PATH.write_text(json.dumps(parent_check,indent=2,ensure_ascii=False),encoding="utf-8")
print(json.dumps(parent_check,indent=2,ensure_ascii=False))
if not PARENT_EQUIVALENCE_PASSED:
    raise ValueError("Uploaded Branch B representation is not exactly reproducible from the frozen D6 source.")


{
  "document_id": "D6",
  "branch": "C",
  "parent_branch": "B",
  "source_sha256": "b3cf121cac62f6807d41e4e013a6edeebec0cbbf4f0ea2995e1fcc135abaac8d",
  "branch_B_conversion_integrity_passed": true,
  "uploaded_branch_B_sha256": "fc2716f7590f8fdb9eca5b304dfe202670682ff697bfb80efa75f2bf67e51d46",
  "reproduced_branch_B_sha256": "fc2716f7590f8fdb9eca5b304dfe202670682ff697bfb80efa75f2bf67e51d46",
  "branch_B_representation_exactly_reproduced": true,
  "source_page_count": 10,
  "parent_equivalence_passed": true
}


In [38]:
# ============================================================
# 5. Conservative deterministic normalisation
# ============================================================
UNICODE_SPACES=["\u00a0","\u1680","\u2000","\u2001","\u2002","\u2003","\u2004","\u2005",
                "\u2006","\u2007","\u2008","\u2009","\u200a","\u202f","\u205f","\u3000"]
APOSTROPHES={"’":"'","‘":"'","‛":"'","´":"'","`":"'"}
DASHES={"‐":"-","‑":"-","‒":"-","–":"-","—":"-","−":"-"}

def normalise_text_representation(text):
    text=unicodedata.normalize("NFKC",str(text))
    for c in UNICODE_SPACES: text=text.replace(c," ")
    for a,b in APOSTROPHES.items(): text=text.replace(a,b)
    for a,b in DASHES.items(): text=text.replace(a,b)
    text=text.replace("\u00ad","").replace("\r\n","\n").replace("\r","\n")
    lines=[re.sub(r"[ \t\f\v]+"," ",line).rstrip() for line in text.splitlines()]
    text="\n".join(lines)
    text=re.sub(r"\n{3,}","\n\n",text)
    return text.strip()+"\n"

NORMALISED_MARKDOWN=normalise_text_representation(SOURCE_B_MARKDOWN)
if not NORMALISED_MARKDOWN.strip():
    raise ValueError("Branch C produced an empty representation.")
print("B chars:",len(SOURCE_B_MARKDOWN),"C chars:",len(NORMALISED_MARKDOWN))


B chars: 19111 C chars: 19022


In [39]:
# ============================================================
# 6. Transformation-aware Branch C normalisation integrity
# ============================================================
PAGE_PATTERN=re.compile(r"^## Source Page (\d+)$",flags=re.MULTILINE)
parent_pages=PAGE_PATTERN.findall(SOURCE_B_MARKDOWN)
branch_c_pages=PAGE_PATTERN.findall(NORMALISED_MARKDOWN)
expected_pages=[str(i) for i in range(1,EXPECTED_PAGE_COUNT+1)]
page_sequence_preserved=(parent_pages==branch_c_pages==expected_pages)

EXPECTED_NORMALISED_MARKDOWN=normalise_text_representation(SOURCE_B_MARKDOWN)
deterministic_representation_verified=(NORMALISED_MARKDOWN==EXPECTED_NORMALISED_MARKDOWN)

canon=NORMALISED_MARKDOWN.casefold()
component_checks={}
for key,marker in EXPECTED_COMPONENT_MARKERS.items():
    cm=normalise_text_representation(marker).strip().casefold()
    component_checks[key]=(cm in canon)
all_expected_components_preserved=all(component_checks.values())

VALUE_PATTERNS={
    "currency_amounts":r"\$\s*\d[\d,]*(?:\.\d+)?",
    "percentages":r"(?<![\w])\(?-?\d+(?:\.\d+)?\)?\s*%",
    "billion_million_expressions":r"\$\s*\d+(?:\.\d+)?\s*(?:billion|million)\b",
    "comma_separated_numbers":r"(?<![\w])\(?-?\d{1,3}(?:,\d{3})+(?:\.\d+)?\)?(?![\w])",
    "per_share_values":r"\$\s*\d+\.\d{2}\b",
    "date_periods":r"\b(?:September|June)\s+30,\s+202[23]\b"
}
def canon_token(t):
    t=normalise_text_representation(t).strip()
    t=re.sub(r"\$\s+","$",t)
    return re.sub(r"[ \t]+"," ",t).casefold()

numeric_preservation={}
for label,pattern in VALUE_PATTERNS.items():
    before=[canon_token(x) for x in re.findall(pattern,SOURCE_B_MARKDOWN,flags=re.I)]
    after=[canon_token(x) for x in re.findall(pattern,NORMALISED_MARKDOWN,flags=re.I)]
    bc,ac=Counter(before),Counter(after)
    missing=list((bc-ac).elements()); added=list((ac-bc).elements())
    numeric_preservation[label]={
        "count_before":len(before),"count_after":len(after),
        "missing_token_count":len(missing),"added_token_count":len(added),
        "passed":len(missing)==0 and len(added)==0
    }
numeric_values_preserved=all(x["passed"] for x in numeric_preservation.values())

neg_before=re.findall(r"\(\s*\d[\d,]*(?:\.\d+)?\s*\)",SOURCE_B_MARKDOWN)
neg_after=re.findall(r"\(\s*\d[\d,]*(?:\.\d+)?\s*\)",NORMALISED_MARKDOWN)
negative_parentheses_preserved=(Counter(neg_before)==Counter(neg_after))

zero_before=len(re.findall(r"(?<![\d.])0(?![\d.])",SOURCE_B_MARKDOWN))
zero_after=len(re.findall(r"(?<![\d.])0(?![\d.])",NORMALISED_MARKDOWN))
zero_value_count_preserved=(zero_before==zero_after)

scope_page_presence={
    str(part):{str(page):f"## Source Page {page}" in NORMALISED_MARKDOWN for page in pages}
    for part,pages in PART_PAGE_RANGES.items()
}
all_scope_pages_present=all(all(v.values()) for v in scope_page_presence.values())

normalisation_integrity_passed=bool(
    PARENT_EQUIVALENCE_PASSED and page_sequence_preserved and
    deterministic_representation_verified and all_expected_components_preserved and
    numeric_values_preserved and negative_parentheses_preserved and
    zero_value_count_preserved and all_scope_pages_present
)

normalisation_check={
    "document_id":DOCUMENT_ID,"branch":BRANCH,"parent_branch":PARENT_BRANCH,
    "parent_equivalence_passed":PARENT_EQUIVALENCE_PASSED,
    "parent_page_markers":parent_pages,"branch_C_page_markers":branch_c_pages,
    "page_sequence_preserved":page_sequence_preserved,
    "deterministic_representation_verified":deterministic_representation_verified,
    "component_checks":component_checks,
    "all_expected_components_preserved":all_expected_components_preserved,
    "numeric_token_preservation":numeric_preservation,
    "numeric_values_preserved":numeric_values_preserved,
    "negative_parentheses_count_before":len(neg_before),
    "negative_parentheses_count_after":len(neg_after),
    "negative_parentheses_preserved":negative_parentheses_preserved,
    "standalone_zero_count_before":zero_before,
    "standalone_zero_count_after":zero_after,
    "zero_value_count_preserved":zero_value_count_preserved,
    "scope_page_presence":scope_page_presence,
    "all_scope_pages_present":all_scope_pages_present,
    "complete_10_page_representation_retained":True,
    "same_complete_representation_used_for_all_parts":True,
    "part_specific_source_filtering_applied":False,
    "page_cropping_applied":False,"page_removal_applied":False,
    "unicode_nfkc_normalisation_applied":True,
    "unicode_space_standardisation_applied":True,
    "apostrophe_standardisation_applied":True,
    "dash_and_minus_standardisation_applied":True,
    "soft_hyphen_removal_applied":True,
    "line_endings_standardised":True,
    "horizontal_whitespace_normalisation_applied":True,
    "paragraph_line_merging_applied":False,
    "line_break_hyphenation_repair_applied":False,
    "semantic_harmonisation_applied":False,
    "semantic_rewriting_applied":False,
    "unit_conversion_applied":False,
    "numeric_calculation_applied":False,
    "manual_correction_applied":False,
    "reference_values_used_for_transformation":False,
    "normalisation_integrity_passed":normalisation_integrity_passed
}
NORMALISATION_CHECK_PATH=OUTPUT_DIR/"D6_branch_C_normalisation_check.json"
NORMALISATION_CHECK_PATH.write_text(json.dumps(normalisation_check,indent=2,ensure_ascii=False),encoding="utf-8")
print(json.dumps(normalisation_check,indent=2,ensure_ascii=False))
if not normalisation_integrity_passed:
    raise ValueError("D6 Branch C normalisation-integrity checks failed.")


{
  "document_id": "D6",
  "branch": "C",
  "parent_branch": "B",
  "parent_equivalence_passed": true,
  "parent_page_markers": [
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8",
    "9",
    "10"
  ],
  "branch_C_page_markers": [
    "1",
    "2",
    "3",
    "4",
    "5",
    "6",
    "7",
    "8",
    "9",
    "10"
  ],
  "page_sequence_preserved": true,
  "deterministic_representation_verified": true,
  "component_checks": {
    "quarterly_results": true,
    "business_highlights": true,
    "financial_performance_reconciliation": true,
    "segment_revenue_reconciliation": true,
    "selected_product_reconciliation": true,
    "income_statements": true,
    "comprehensive_income_statements": true,
    "balance_sheets": true,
    "cash_flows_statements": true,
    "segment_revenue_operating_income": true
  },
  "all_expected_components_preserved": true,
  "numeric_token_preservation": {
    "currency_amounts": {
      "count_before": 66,
      "count_after":

In [40]:
# ============================================================
# 7. Save ONE complete Branch C representation
# ============================================================
REPRESENTATION_PATH=OUTPUT_DIR/"D6_branch_C_normalised_markdown.md"
REPRESENTATION_PATH.write_text(NORMALISED_MARKDOWN,encoding="utf-8")
REPRESENTATION_SHA256=sha256_file(REPRESENTATION_PATH)
print(REPRESENTATION_PATH.name,REPRESENTATION_SHA256)


D6_branch_C_normalised_markdown.md 888a08dd6c811052d9f8eb57f10168b9b7eaad49db543ed022b6754928eb0f68


In [41]:
# ============================================================
# 8. Build five Branch C prompts using the frozen Branch A/B scopes
# ============================================================

COMMON_INSTRUCTIONS = r'''
You are an information extraction assistant.

Extract the requested financial and quantitative records represented
within the defined source scope of the attached deterministically normalised structural Markdown representation of the Microsoft FY24 Q1 Press Release.

Treat the attached deterministically normalised structural Markdown document as the only
source of information.

The complete 10-page source representation is attached. Process only
the source scope defined for the current extraction part.

For every included record return exactly these twelve fields:

- Category
- Statement or Section
- Metric
- Business Area
- Value 2023
- Value 2022
- GAAP YoY Change
- Constant Currency Impact
- Constant Currency YoY Change
- Unit
- Reporting Period
- Source Location

General extraction rules:

- Use JSON numbers for explicitly represented numeric values.
- Use JSON null when a field is not explicitly represented.
- Preserve negative values.
- Preserve explicit zero values.
- Preserve the source measurement scale.
- Preserve repeated observations when they occur in distinct source
  sections.
- Do not calculate or infer values.
- Do not derive prior-period values from growth rates.
- Do not convert units.
- Do not normalise measurement scales.
- Do not silently correct source values.
- Do not use external knowledge.
- Do not include publication metadata, webcast details, contact details,
  URLs, qualitative outlook text or forward-looking risk narrative.
- Do not include values that appear only inside descriptive accounting
  line-item labels when they are not primary observations.
- Ignore Markdown syntax and representation metadata that are not
  source financial observations.

For Balance Sheets only:

- place the September 30, 2023 amount in "Value 2023";
- place the June 30, 2023 amount in "Value 2022".

Return only valid JSON.

Do not include Markdown fences, explanations or commentary.

Return the result using exactly this top-level structure:

{
  "document_id": "D6",
  "branch": "C",
  "part": PART_NUMBER,
  "records": [
    {
      "Category": null,
      "Statement or Section": null,
      "Metric": null,
      "Business Area": null,
      "Value 2023": null,
      "Value 2022": null,
      "GAAP YoY Change": null,
      "Constant Currency Impact": null,
      "Constant Currency YoY Change": null,
      "Unit": null,
      "Reporting Period": null,
      "Source Location": null
    }
  ]
}

Return only the JSON object.
'''.strip()

PART_SCOPES = {
    1: r'''
PART 1 SOURCE SCOPE

Include only:

1. The quantitative company-performance observations represented in the
   quarterly-results and business-highlights sections on page 1 and the
   shareholder-return observation on page 2.

2. Every represented metric in the following three constant-currency
   reconciliation tables on page 3:

   - Financial Performance Constant Currency Reconciliation
   - Segment Revenue Constant Currency Reconciliation
   - Selected Product and Service Revenue Constant Currency Reconciliation

Valid Category values for this part:

- Narrative performance highlight
- Financial performance reconciliation
- Segment revenue reconciliation
- Selected product and service reconciliation

Valid Source Location values:

- Page 1 — Quarterly results
- Page 1 — Business Highlights
- Page 2 — Shareholder returns
- Page 3 — Financial Performance Constant Currency Reconciliation
- Page 3 — Segment Revenue Constant Currency Reconciliation
- Page 3 — Selected Product and Service Revenue Constant Currency Reconciliation

Extract every record represented within this defined source scope.
''',

    2: r'''
PART 2 SOURCE SCOPE

Include every primary financial line item represented in:

- Income Statements on page 6;
- Comprehensive Income Statements on page 7.

Exclude section headings and accounting-label values that are not
primary financial line-item observations.

Valid Category values:

- Income statement
- Comprehensive income statement

Valid Source Location values:

- Page 6 — Income Statements
- Page 7 — Comprehensive Income Statements

Use:

- "Income Statements" or "Comprehensive Income Statements" for
  Statement or Section;
- "Corporate" for Business Area;
- "Three months ended September 30" for Reporting Period.

Extract every record represented within this defined source scope.
''',

    3: r'''
PART 3 SOURCE SCOPE

Include every primary Balance Sheets line item represented on page 8.

Exclude:

- headings without amounts;
- Commitments and contingencies;
- allowance amounts embedded only in the accounts-receivable label;
- accumulated depreciation amounts embedded only in the property and
  equipment label;
- authorised and outstanding share quantities embedded only in the
  common-stock label.

Use:

- Category: "Balance sheet"
- Statement or Section: "Balance Sheets"
- Business Area: "Corporate"
- Unit: "USD millions"
- Reporting Period: "September 30, 2023 and June 30, 2023"
- Source Location: "Page 8 — Balance Sheets"

Place September 30, 2023 values in "Value 2023".

Place June 30, 2023 values in "Value 2022".

Extract every record represented within this defined source scope.
''',

    4: r'''
PART 4 SOURCE SCOPE

Include every primary Cash Flows Statements line item represented on
page 9.

Preserve parentheses as negative values and preserve explicit zeros.

Distinguish the two separate source observations:

- Other, net — financing
- Other, net — investing

Use:

- Category: "Cash flow statement"
- Statement or Section: "Cash Flows Statements"
- Business Area: "Corporate"
- Unit: "USD millions"
- Reporting Period: "Three months ended September 30"
- Source Location: "Page 9 — Cash Flows Statements"

Extract every record represented within this defined source scope.
''',

    5: r'''
PART 5 SOURCE SCOPE

Include every represented revenue and operating-income row from the
Segment Revenue and Operating Income table on page 10.

The table contains revenue observations and operating-income
observations for the represented business segments and total rows.

Use:

- Category: "Segment revenue and operating income"
- Statement or Section: "Segment Revenue and Operating Income"
- Unit: "USD millions"
- Reporting Period: "Three months ended September 30"
- Source Location: "Page 10 — Segment Revenue and Operating Income"

Extract every record represented within this defined source scope.
'''
}

PART_PROMPT_PATHS = {}

for part_number, part_scope in PART_SCOPES.items():
    prompt = (
        COMMON_INSTRUCTIONS.replace(
            "PART_NUMBER",
            str(part_number)
        )
        + "\n\n"
        + part_scope.strip()
    )

    path = (
        OUTPUT_DIR
        / f"D6_branch_C_prompt_part_{part_number}.txt"
    )

    path.write_text(
        prompt,
        encoding="utf-8"
    )

    PART_PROMPT_PATHS[part_number] = path

    print(
        f"Part {part_number}:",
        path.name,
        "| SHA-256:",
        sha256_file(path)
    )


Part 1: D6_branch_C_prompt_part_1.txt | SHA-256: acc376f3134e1083ed5183b3822d017d9287394a6d03db57cabfce74b701c4c8
Part 2: D6_branch_C_prompt_part_2.txt | SHA-256: 029feb84b26b26aa1eab1d5e38b296337109a41445e0d9c8e59c3f977a51e421
Part 3: D6_branch_C_prompt_part_3.txt | SHA-256: 04a33d758d14d6f3842075aa0eaa79851b62342d30ab78fc7e183226cec88f11
Part 4: D6_branch_C_prompt_part_4.txt | SHA-256: dc358448d90add4f4e16a7e8da58cdd376ca907e824484cf63ab7d33febc74f0
Part 5: D6_branch_C_prompt_part_5.txt | SHA-256: f2ddc3f5513638ef7c68892fae08e26a22d42eff567c8029682db37b1af6cba9


In [42]:
# ============================================================
# 9. Metadata + pre-extraction control
# ============================================================
REPRESENTATION_METADATA={
    "document_id":DOCUMENT_ID,"branch":BRANCH,"parent_branch":PARENT_BRANCH,
    "source_file":SOURCE_PATH.name,"source_sha256":SOURCE_SHA256,
    "parent_B_representation_file":BRANCH_B_REPRESENTATION_PATH.name,
    "parent_B_representation_sha256":SOURCE_B_SHA256,
    "parent_B_equivalence_passed":PARENT_EQUIVALENCE_PASSED,
    "representation_type":"Complete Branch B structural Markdown with deterministic normalisation",
    "representation_file":REPRESENTATION_PATH.name,
    "representation_sha256":REPRESENTATION_SHA256,
    "complete_10_page_representation_retained":True,
    "same_complete_representation_used_for_all_parts":True,
    "part_specific_source_filtering_applied":False,
    "structural_conversion_inherited_from_branch_B":True,
    "normalisation_applied":True,
    "normalisation_integrity_passed":normalisation_check["normalisation_integrity_passed"],
    "reference_values_used_for_transformation":False
}
REP_METADATA_PATH=OUTPUT_DIR/"D6_branch_C_representation_metadata.json"
REP_METADATA_PATH.write_text(json.dumps(REPRESENTATION_METADATA,indent=2,ensure_ascii=False),encoding="utf-8")

EXPERIMENT_METADATA_PRE={
    "document_id":DOCUMENT_ID,"document_name":DOCUMENT_NAME,"branch":BRANCH,
    "branch_name":BRANCH_NAME,"parent_branch":PARENT_BRANCH,
    "source_file":SOURCE_PATH.name,"source_sha256":SOURCE_SHA256,"source_verified":SOURCE_HASH_MATCH,
    "input_representation":"Complete deterministically normalised structural Markdown",
    "representation_file":REPRESENTATION_PATH.name,"representation_sha256":REPRESENTATION_SHA256,
    "parent_B_equivalence_passed":PARENT_EQUIVALENCE_PASSED,
    "normalisation_integrity_passed":normalisation_check["normalisation_integrity_passed"],
    "same_complete_representation_used_for_all_parts":True,
    "split_extraction_applied":True,"split_part_count":5,
    "reference_values_disclosed_to_model":False,
    "reference_values_used_for_transformation":False,
    "expected_record_count_disclosed_to_model":False,
    "expected_category_counts_disclosed_to_model":False,
    "manual_response_repair_permitted":False,
    "model":"GPT-5.5",
    "execution_environment":"Independent ChatGPT conversations",
    "prompt_files":{
        str(part):{"file":path.name,"sha256":sha256_file(path)}
        for part,path in PART_PROMPT_PATHS.items()
    },
    "created_at":datetime.now().isoformat()
}
METADATA_PRE_PATH=OUTPUT_DIR/"D6_branch_C_experiment_metadata_pre.json"
METADATA_PRE_PATH.write_text(json.dumps(EXPERIMENT_METADATA_PRE,indent=2,ensure_ascii=False),encoding="utf-8")

PRECHECK={
    "document_id":DOCUMENT_ID,"branch":BRANCH,
    "source_identity_verified":SOURCE_HASH_MATCH,
    "parent_B_equivalence_passed":PARENT_EQUIVALENCE_PASSED,
    "normalisation_integrity_passed":normalisation_check["normalisation_integrity_passed"],
    "complete_10_page_representation_retained":True,
    "same_complete_representation_used_for_all_parts":True,
    "page_sequence_preserved":page_sequence_preserved,
    "all_scope_pages_present":all_scope_pages_present,
    "representation_exists":REPRESENTATION_PATH.exists(),
    "all_five_prompts_exist":all(p.exists() for p in PART_PROMPT_PATHS.values()),
    "expected_record_count_disclosed_to_model":False,
    "expected_category_counts_disclosed_to_model":False,
    "reference_values_used_for_transformation":False
}
PRECHECK["ready_for_independent_llm_execution"]=bool(
    PRECHECK["source_identity_verified"] and PRECHECK["parent_B_equivalence_passed"] and
    PRECHECK["normalisation_integrity_passed"] and PRECHECK["representation_exists"] and
    PRECHECK["all_five_prompts_exist"]
)
PRECHECK_PATH=OUTPUT_DIR/"D6_branch_C_pre_extraction_check.json"
PRECHECK_PATH.write_text(json.dumps(PRECHECK,indent=2,ensure_ascii=False),encoding="utf-8")
print(json.dumps(PRECHECK,indent=2,ensure_ascii=False))
if not PRECHECK["ready_for_independent_llm_execution"]:
    raise ValueError("D6 Branch C is not ready for independent extraction.")


{
  "document_id": "D6",
  "branch": "C",
  "source_identity_verified": true,
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "complete_10_page_representation_retained": true,
  "same_complete_representation_used_for_all_parts": true,
  "page_sequence_preserved": true,
  "all_scope_pages_present": true,
  "representation_exists": true,
  "all_five_prompts_exist": true,
  "expected_record_count_disclosed_to_model": false,
  "expected_category_counts_disclosed_to_model": false,
  "reference_values_used_for_transformation": false,
  "ready_for_independent_llm_execution": true
}


In [43]:
# ============================================================
# 10. Download pre-extraction artefacts
# ============================================================
for path in [
    PARENT_CHECK_PATH,NORMALISATION_CHECK_PATH,REPRESENTATION_PATH,
    REP_METADATA_PATH,METADATA_PRE_PATH,PRECHECK_PATH,*PART_PROMPT_PATHS.values()
]:
    files.download(path)
print("Use the SAME D6_branch_C_normalised_markdown.md in all five independent runs.")
print("Submit only the corresponding part prompt. Do not upload reference values.")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Use the SAME D6_branch_C_normalised_markdown.md in all five independent runs.
Submit only the corresponding part prompt. Do not upload reference values.


In [44]:
# ============================================================
# 9. Upload and preserve five raw Branch C responses
# ============================================================

uploaded_outputs = files.upload()

uploaded_txt_paths = [
    Path(name)
    for name in uploaded_outputs
    if name.lower().endswith(".txt")
]

if len(uploaded_txt_paths) != 5:
    raise ValueError(
        "Upload exactly five TXT raw-response files."
    )

def detect_part_number(path):
    match = re.search(
        r"part[_\- ]?([1-5])",
        path.stem,
        flags=re.IGNORECASE
    )

    if match is None:
        raise ValueError(
            f"Could not identify part number from {path.name}."
        )

    return int(match.group(1))

PART_RAW_RESPONSE_PATHS = {}

for uploaded_path in uploaded_txt_paths:
    part_number = detect_part_number(uploaded_path)

    if part_number in PART_RAW_RESPONSE_PATHS:
        raise ValueError(
            f"More than one response was detected for part {part_number}."
        )

    raw_text = uploaded_path.read_text(
        encoding="utf-8"
    )

    if not raw_text.strip():
        raise ValueError(
            f"Raw response for part {part_number} is empty."
        )

    preserved_path = (
        OUTPUT_DIR
        / f"D6_branch_C_raw_response_part_{part_number}.txt"
    )

    preserved_path.write_text(
        raw_text,
        encoding="utf-8"
    )

    PART_RAW_RESPONSE_PATHS[part_number] = preserved_path

if set(PART_RAW_RESPONSE_PATHS) != set(EXPECTED_PART_COUNTS):
    raise ValueError(
        "Uploaded files must represent parts 1, 2, 3, 4 and 5."
    )

PART_RAW_RESPONSE_METADATA = {
    str(part_number): {
        "file": path.name,
        "sha256": sha256_file(path),
        "character_count":
            len(path.read_text(encoding="utf-8"))
    }
    for part_number, path
    in sorted(PART_RAW_RESPONSE_PATHS.items())
}

print(json.dumps(
    PART_RAW_RESPONSE_METADATA,
    ensure_ascii=False,
    indent=2
))


Saving D6_branch_C_raw_response_part_5.txt to D6_branch_C_raw_response_part_5.txt
Saving D6_branch_C_raw_response_part_4.txt to D6_branch_C_raw_response_part_4.txt
Saving D6_branch_C_raw_response_part_3.txt to D6_branch_C_raw_response_part_3.txt
Saving D6_branch_C_raw_response_part_2.txt to D6_branch_C_raw_response_part_2.txt
Saving D6_branch_C_raw_response_part_1.txt to D6_branch_C_raw_response_part_1.txt
{
  "1": {
    "file": "D6_branch_C_raw_response_part_1.txt",
    "sha256": "0f068c536a4d66ed2024cf0e5d399bbdabbac2ea3ef12fe205285cbac4487fe5",
    "character_count": 22033
  },
  "2": {
    "file": "D6_branch_C_raw_response_part_2.txt",
    "sha256": "58225cf30235e43a1404b13cabf57bb38d8c7fd1074aeb555fe8603d7d67697c",
    "character_count": 10432
  },
  "3": {
    "file": "D6_branch_C_raw_response_part_3.txt",
    "sha256": "ce85280fbfd9d34aa7d8e85e0cabef16cca5b70ea30c947920d23ddc542af4cf",
    "character_count": 14129
  },
  "4": {
    "file": "D6_branch_C_raw_response_part_4.txt",


In [45]:
# ============================================================
# 10. Parse the five raw responses independently
# ============================================================

part_parsing_results = {}
part_records = {}
combined_records = []

all_parts_json_valid = True
all_parts_records_evaluable = True

for part_number in sorted(PART_RAW_RESPONSE_PATHS):
    raw_path = PART_RAW_RESPONSE_PATHS[part_number]
    raw_text = raw_path.read_text(encoding="utf-8")

    valid_json = False
    parsing_error = None
    parsed_part = None

    try:
        parsed_part = json.loads(raw_text)
        valid_json = True
    except json.JSONDecodeError as error:
        parsing_error = str(error)

    top_level_object_valid = (
        valid_json and isinstance(parsed_part, dict)
    )

    document_id_present = (
        top_level_object_valid
        and "document_id" in parsed_part
    )
    document_id_correct = (
        document_id_present
        and parsed_part.get("document_id") == DOCUMENT_ID
    )

    branch_present = (
        top_level_object_valid
        and "branch" in parsed_part
    )
    branch_correct = (
        branch_present
        and parsed_part.get("branch") == BRANCH
    )

    part_present = (
        top_level_object_valid
        and "part" in parsed_part
    )
    part_correct = (
        part_present
        and parsed_part.get("part") == part_number
    )

    records_present = (
        top_level_object_valid
        and "records" in parsed_part
    )
    records_is_list = (
        records_present
        and isinstance(parsed_part.get("records"), list)
    )

    records_evaluable = all([
        valid_json,
        top_level_object_valid,
        document_id_present,
        document_id_correct,
        branch_present,
        branch_correct,
        part_present,
        part_correct,
        records_present,
        records_is_list
    ])

    records = (
        parsed_part["records"]
        if records_evaluable
        else []
    )

    observed_part_count = (
        len(records)
        if records_evaluable
        else None
    )

    part_record_count_matches = (
        observed_part_count == EXPECTED_PART_COUNTS[part_number]
        if records_evaluable
        else None
    )

    part_records[part_number] = records

    part_parsing_results[part_number] = {
        "raw_response_file": raw_path.name,
        "raw_response_sha256": sha256_file(raw_path),
        "json_valid": valid_json,
        "json_parsing_error": parsing_error,
        "top_level_object_valid": top_level_object_valid,
        "document_id_present": document_id_present,
        "document_id_correct": document_id_correct,
        "branch_present": branch_present,
        "branch_correct": branch_correct,
        "part_present": part_present,
        "part_correct": part_correct,
        "records_present": records_present,
        "records_is_list": records_is_list,
        "records_evaluable": records_evaluable,
        "expected_record_count":
            EXPECTED_PART_COUNTS[part_number],
        "observed_record_count":
            observed_part_count,
        "record_count_matches":
            part_record_count_matches
    }

    if records_evaluable:
        combined_records.extend(records)

    if not valid_json:
        all_parts_json_valid = False

    if not records_evaluable:
        all_parts_records_evaluable = False

PART_EXECUTION_SUMMARY = {
    str(part_number): result
    for part_number, result
    in part_parsing_results.items()
}

PART_EXECUTION_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D6_branch_C_part_execution_summary.json"
)

PART_EXECUTION_SUMMARY_PATH.write_text(
    json.dumps(
        PART_EXECUTION_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

observed_record_count = (
    len(combined_records)
    if all_parts_records_evaluable
    else None
)

print(json.dumps(
    PART_EXECUTION_SUMMARY,
    ensure_ascii=False,
    indent=2
))
print("All parts JSON valid:", all_parts_json_valid)
print("All parts records evaluable:", all_parts_records_evaluable)
print("Combined observed records:", observed_record_count)


{
  "1": {
    "raw_response_file": "D6_branch_C_raw_response_part_1.txt",
    "raw_response_sha256": "0f068c536a4d66ed2024cf0e5d399bbdabbac2ea3ef12fe205285cbac4487fe5",
    "json_valid": true,
    "json_parsing_error": null,
    "top_level_object_valid": true,
    "document_id_present": true,
    "document_id_correct": true,
    "branch_present": true,
    "branch_correct": true,
    "part_present": true,
    "part_correct": true,
    "records_present": true,
    "records_is_list": true,
    "records_evaluable": true,
    "expected_record_count": 46,
    "observed_record_count": 46,
    "record_count_matches": true
  },
  "2": {
    "raw_response_file": "D6_branch_C_raw_response_part_2.txt",
    "raw_response_sha256": "58225cf30235e43a1404b13cabf57bb38d8c7fd1074aeb555fe8603d7d67697c",
    "json_valid": true,
    "json_parsing_error": null,
    "top_level_object_valid": true,
    "document_id_present": true,
    "document_id_correct": true,
    "branch_present": true,
    "branch_corre

In [46]:
# ============================================================
# 11. Validate combined record schema and field types
# ============================================================

record_structure_issues = []
field_type_issues = []
missing_mandatory_values = []

extracted_records = (
    combined_records
    if all_parts_records_evaluable
    else []
)

for record_index, record in enumerate(extracted_records):
    if not isinstance(record, dict):
        record_structure_issues.append({
            "record_index": record_index,
            "issue": "Record is not a JSON object"
        })
        continue

    actual_fields = set(record.keys())
    expected_fields = set(EXPECTED_FIELDS)

    missing_fields = sorted(
        expected_fields - actual_fields
    )
    extra_fields = sorted(
        actual_fields - expected_fields
    )

    if missing_fields or extra_fields:
        record_structure_issues.append({
            "record_index": record_index,
            "missing_fields": missing_fields,
            "extra_fields": extra_fields
        })

    for field in STRING_OR_NULL_FIELDS:
        value = record.get(field)
        if value is not None and not isinstance(value, str):
            field_type_issues.append({
                "record_index": record_index,
                "field": field,
                "observed_type": type(value).__name__
            })

    for field in NUMERIC_OR_NULL_FIELDS:
        value = record.get(field)
        if (
            value is not None
            and (
                isinstance(value, bool)
                or not isinstance(value, (int, float))
            )
        ):
            field_type_issues.append({
                "record_index": record_index,
                "field": field,
                "observed_type": type(value).__name__,
                "observed_value": value
            })

    missing_content = [
        field
        for field in MANDATORY_CONTENT_FIELDS
        if record.get(field) is None
    ]

    if missing_content:
        missing_mandatory_values.append({
            "record_index": record_index,
            "missing_mandatory_fields": missing_content
        })

record_schema_valid = (
    len(record_structure_issues) == 0
    if all_parts_records_evaluable
    else None
)

field_types_valid = (
    len(field_type_issues) == 0
    if all_parts_records_evaluable
    else None
)

mandatory_fields_complete = (
    len(missing_mandatory_values) == 0
    if all_parts_records_evaluable
    else None
)

print("Record structure issues:", len(record_structure_issues))
print("Field type issues:", len(field_type_issues))
print("Records with missing mandatory fields:", len(missing_mandatory_values))


Record structure issues: 0
Field type issues: 0
Records with missing mandatory fields: 0


In [48]:
# ============================================================
# 12. Content/scope diagnostics kept separate from schema validity
# ============================================================
#
# IMPORTANT:
# Repeated values under the coarse identity fields used below are
# NOT automatically duplicate extraction records.
#
# D6 legitimately contains repeated labels such as Product,
# Service and other, Basic, and Diluted within different financial
# substructures.
#
# Therefore this check is retained only as a diagnostic of
# non-unique coarse identifiers. Formal one-to-one record identity
# is resolved during Stage 4 validation using the frozen D6
# alignment framework.
# ============================================================

if all_parts_records_evaluable:

    observed_category_counts = dict(
        Counter(
            record.get("Category")
            for record in extracted_records
            if isinstance(record, dict)
        )
    )

    record_count_valid = (
        len(extracted_records)
        == EXPECTED_RECORD_COUNT
    )

    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )

    # --------------------------------------------------------
    # Coarse identity diagnostic only
    # --------------------------------------------------------

    coarse_identity_fields = [
        "Category",
        "Statement or Section",
        "Metric",
        "Business Area",
        "Reporting Period",
        "Source Location"
    ]

    coarse_identity_counter = Counter(
        tuple(
            record.get(field)
            for field in coarse_identity_fields
        )
        for record in extracted_records
        if isinstance(record, dict)
    )

    nonunique_coarse_identity_keys = [
        {
            "key": list(key),
            "occurrence_count": count
        }
        for key, count
        in coarse_identity_counter.items()
        if count > 1
    ]

    nonunique_coarse_identity_key_count = len(
        nonunique_coarse_identity_keys
    )

else:

    observed_category_counts = None

    record_count_valid = None

    category_counts_valid = None

    nonunique_coarse_identity_keys = None

    nonunique_coarse_identity_key_count = None


# ------------------------------------------------------------
# Schema validity
# ------------------------------------------------------------
#
# These variables are created in the previous validation cell:
#
#   record_schema_valid
#   field_types_valid
#
# Expected record counts and category counts are deliberately
# excluded from schema validity.
# ------------------------------------------------------------

schema_validity = bool(
    all_parts_records_evaluable
    and record_schema_valid is True
    and field_types_valid is True
)


# ------------------------------------------------------------
# Scope completeness
# ------------------------------------------------------------

scope_complete = bool(
    record_count_valid
    and category_counts_valid
) if all_parts_records_evaluable else False


# ------------------------------------------------------------
# Aggregate content/scope diagnostics
# ------------------------------------------------------------

CONTENT_DIAGNOSTICS = {

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        (
            len(extracted_records)
            if all_parts_records_evaluable
            else None
        ),

    "record_count_matches_reference":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match_reference":
        category_counts_valid,

    "expected_part_counts":
        EXPECTED_PART_COUNTS,

    "part_execution_results":
        PART_EXECUTION_SUMMARY,

    # --------------------------------------------------------
    # Diagnostic only — not interpreted as duplicate records
    # --------------------------------------------------------

    "nonunique_coarse_identity_key_count":
        nonunique_coarse_identity_key_count,

    "nonunique_coarse_identity_keys":
        nonunique_coarse_identity_keys,

    "coarse_identity_check_is_diagnostic_only":
        True,

    "record_identity_resolved_during_validation":
        True,

    # --------------------------------------------------------
    # Mandatory-field diagnostic
    # --------------------------------------------------------

    "mandatory_fields_complete":
        mandatory_fields_complete,

    # --------------------------------------------------------
    # Technical/schema diagnostic
    # --------------------------------------------------------

    "record_schema_valid":
        record_schema_valid,

    "field_types_valid":
        field_types_valid,

    "schema_validity":
        schema_validity,

    "scope_complete":
        scope_complete
}


print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)

{
  "expected_record_count": 147,
  "observed_record_count": 147,
  "record_count_matches_reference": true,
  "expected_category_counts": {
    "Narrative performance highlight": 24,
    "Financial performance reconciliation": 4,
    "Segment revenue reconciliation": 3,
    "Selected product and service reconciliation": 15,
    "Income statement": 19,
    "Comprehensive income statement": 6,
    "Balance sheet": 34,
    "Cash flow statement": 34,
    "Segment revenue and operating income": 8
  },
  "observed_category_counts": {
    "Narrative performance highlight": 24,
    "Financial performance reconciliation": 4,
    "Segment revenue reconciliation": 3,
    "Selected product and service reconciliation": 15,
    "Income statement": 19,
    "Comprehensive income statement": 6,
    "Balance sheet": 34,
    "Cash flow statement": 34,
    "Segment revenue and operating income": 8
  },
  "category_counts_match_reference": true,
  "expected_part_counts": {
    "1": 46,
    "2": 25,
    "3"

In [49]:
# ============================================================
# 13. Create structural/schema diagnostics
# ============================================================

# Technical/schema validity is intentionally independent of
# expected record counts and expected category counts.

structure_valid = all([
    all_parts_json_valid,
    all_parts_records_evaluable,
    record_schema_valid is True,
    field_types_valid is True
])

STRUCTURE_CHECK = {
    "document_id": DOCUMENT_ID,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "input_representation": "Complete deterministically normalised structural Markdown",
    "all_parts_json_valid": bool(all_parts_json_valid),
    "all_parts_records_evaluable":
        bool(all_parts_records_evaluable),
    "record_schema_valid":
        record_schema_valid,
    "field_types_valid":
        field_types_valid,
    "records_with_structure_issues":
        len(record_structure_issues)
        if all_parts_records_evaluable
        else None,
    "record_structure_issues":
        record_structure_issues
        if all_parts_records_evaluable
        else None,
    "records_with_type_issues":
        len({
            issue["record_index"]
            for issue in field_type_issues
        })
        if all_parts_records_evaluable
        else None,
    "field_type_issues":
        field_type_issues
        if all_parts_records_evaluable
        else None,
    "mandatory_fields_complete":
        mandatory_fields_complete,
    "missing_mandatory_values":
        missing_mandatory_values
        if all_parts_records_evaluable
        else None,
    "content_diagnostics":
        CONTENT_DIAGNOSTICS,
    "structure_valid": bool(structure_valid),
    "scope_complete":
        bool(record_count_valid and category_counts_valid)
}

STRUCTURE_CHECK_PATH = (
    OUTPUT_DIR
    / "D6_branch_C_structure_check.json"
)

STRUCTURE_CHECK_PATH.write_text(
    json.dumps(
        STRUCTURE_CHECK,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    STRUCTURE_CHECK,
    ensure_ascii=False,
    indent=2
))


{
  "document_id": "D6",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "input_representation": "Complete deterministically normalised structural Markdown",
  "all_parts_json_valid": true,
  "all_parts_records_evaluable": true,
  "record_schema_valid": true,
  "field_types_valid": true,
  "records_with_structure_issues": 0,
  "record_structure_issues": [],
  "records_with_type_issues": 0,
  "field_type_issues": [],
  "mandatory_fields_complete": true,
  "missing_mandatory_values": [],
  "content_diagnostics": {
    "expected_record_count": 147,
    "observed_record_count": 147,
    "record_count_matches_reference": true,
    "expected_category_counts": {
      "Narrative performance highlight": 24,
      "Financial performance reconciliation": 4,
      "Segment revenue reconciliation": 3,
      "Selected product and service reconciliation": 15,
      "Income statement": 19,
      "Comprehensive income statement": 6,
      "Balance sheet": 34,
      "Cash flow statem

In [50]:
# ============================================================
# 14. Preserve combined parsed extraction only if all parts are evaluable
# ============================================================

COMBINED_EXTRACTION_PATH = (
    OUTPUT_DIR
    / "D6_branch_C_combined_parsed_extraction.json"
)

combined_extraction_created = False
combined_extraction_sha256 = None

if all_parts_records_evaluable:
    COMBINED_EXTRACTION = {
        "document_id": DOCUMENT_ID,
        "branch": BRANCH,
        "records": extracted_records
    }

    COMBINED_EXTRACTION_PATH.write_text(
        json.dumps(
            COMBINED_EXTRACTION,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )

    combined_extraction_created = True
    combined_extraction_sha256 = sha256_file(
        COMBINED_EXTRACTION_PATH
    )

    print("Combined parsed extraction saved.")
else:
    print(
        "Combined parsed extraction was not created because "
        "one or more parts are not evaluable."
    )


Combined parsed extraction saved.


In [51]:
# ============================================================
# 15. Create final experiment metadata and summary
# ============================================================

EXPERIMENT_METADATA = {
    **EXPERIMENT_METADATA_PRE,
    "raw_response_files":
        PART_RAW_RESPONSE_METADATA,
    "part_execution_summary_file":
        PART_EXECUTION_SUMMARY_PATH.name,
    "part_parsing_results":
        PART_EXECUTION_SUMMARY,
    "all_parts_json_valid":
        all_parts_json_valid,
    "all_parts_records_evaluable":
        all_parts_records_evaluable,
    "combined_parsed_extraction_file":
        COMBINED_EXTRACTION_PATH.name
        if combined_extraction_created
        else None,
    "combined_parsed_extraction_sha256":
        combined_extraction_sha256,
    "observed_record_count":
        len(extracted_records)
        if all_parts_records_evaluable
        else None,
    "observed_category_counts":
        observed_category_counts,
    "structure_check_file":
        STRUCTURE_CHECK_PATH.name,
    "structure_valid":
        bool(structure_valid),
    "notes": (
        "Branch C uses the same complete converted 10-page Markdown "
        "representation in all five predefined extraction runs. "
        "Only the scope prompt changes between parts, matching the "
        "five-part Branch A execution protocol. Stage 1 expected "
        "counts are retained only for post-extraction diagnostics "
        "and are not disclosed to the model."
    )
}

EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR
    / "D6_branch_C_experiment_metadata.json"
)

EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

EXPERIMENT_SUMMARY = {
    "document_id": DOCUMENT_ID,
    "document_name": DOCUMENT_NAME,
    "branch": BRANCH,
    "branch_name": BRANCH_NAME,
    "source_file": SOURCE_PATH.name,
    "source_sha256": SOURCE_SHA256,
    "source_verified": SOURCE_HASH_MATCH,
    "input_representation": "Complete deterministically normalised structural Markdown",
    "representation_file":
        REPRESENTATION_PATH.name,
    "representation_sha256":
        REPRESENTATION_SHA256,
    "parent_B_equivalence_passed": PARENT_EQUIVALENCE_PASSED,
    "normalisation_integrity_passed": normalisation_check["normalisation_integrity_passed"],
    "structural_conversion_applied": True,
    "same_complete_representation_used_for_all_parts": True,
    "split_extraction_applied": True,
    "split_part_count": 5,
    "all_parts_json_valid":
        bool(all_parts_json_valid),
    "all_parts_records_evaluable":
        bool(all_parts_records_evaluable),
    "expected_record_count":
        EXPECTED_RECORD_COUNT,
    "observed_record_count":
        len(extracted_records)
        if all_parts_records_evaluable
        else None,
    "record_count_matches":
        record_count_valid,
    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,
    "observed_category_counts":
        observed_category_counts,
    "category_counts_match":
        category_counts_valid,
    "structure_valid":
        bool(structure_valid),
    "scope_complete":
        bool(record_count_valid and category_counts_valid),
    "combined_parsed_extraction_created":
        combined_extraction_created,
    "expected_record_count_disclosed_to_model": False,
    "expected_category_counts_disclosed_to_model": False,
    "reference_values_used_for_transformation": False,
    "accuracy_validation_completed": False,
    "validation_status": (
        "Pending Stage 4 Branch C validation against the fixed Stage 1 reference dataset using Branch A-frozen D6 comparison rules"
        if combined_extraction_created
        else
        "Not fully content-evaluable because one or more Branch C parts lack an evaluable records structure"
    )
}

EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D6_branch_C_experiment_summary.json"
)

EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

print(json.dumps(
    EXPERIMENT_SUMMARY,
    ensure_ascii=False,
    indent=2
))


{
  "document_id": "D6",
  "document_name": "Microsoft FY24 Q1 Press Release",
  "branch": "C",
  "branch_name": "Deterministic normalisation",
  "source_file": "D6 - Microsoft FY24 Q1 Press Release.pdf",
  "source_sha256": "b3cf121cac62f6807d41e4e013a6edeebec0cbbf4f0ea2995e1fcc135abaac8d",
  "source_verified": true,
  "input_representation": "Complete deterministically normalised structural Markdown",
  "representation_file": "D6_branch_C_normalised_markdown.md",
  "representation_sha256": "888a08dd6c811052d9f8eb57f10168b9b7eaad49db543ed022b6754928eb0f68",
  "parent_B_equivalence_passed": true,
  "normalisation_integrity_passed": true,
  "structural_conversion_applied": true,
  "same_complete_representation_used_for_all_parts": true,
  "split_extraction_applied": true,
  "split_part_count": 5,
  "all_parts_json_valid": true,
  "all_parts_records_evaluable": true,
  "expected_record_count": 147,
  "observed_record_count": 147,
  "record_count_matches": true,
  "expected_category_counts